In [6]:
## Embedding Based Semantic Chunking

import numpy as np

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_chunk(sentences, embed_fn, similarity_threshold=0.75, max_chunk_tokens=300):
    embeddings = [embed_fn(s) for s in sentences]
    chunks, current = [], [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_sim(embeddings[i-1], embeddings[i])
        would_exceed = len(" ".join(current + [sentences[i]]).split()) > max_chunk_tokens
        if sim < similarity_threshold or would_exceed:
            chunks.append(" ".join(current))
            current = [sentences[i]]
        else:
            current.append(sentences[i])

    if current:
        chunks.append(" ".join(current))
    return chunks


In [7]:
## Breakpoint Detection Strategies (fixed / percentile / std-dev)

def compute_similarities(sentences, embed_fn):
    embeddings = [embed_fn(s) for s in sentences]
    return [cosine_sim(embeddings[i-1], embeddings[i]) for i in range(1, len(embeddings))]

def fixed_threshold_breakpoints(similarities, threshold=0.75):
    return [i for i, s in enumerate(similarities) if s < threshold]

def percentile_breakpoints(similarities, percentile=25):
    cutoff = np.percentile(similarities, percentile)
    return [i for i, s in enumerate(similarities) if s < cutoff]

def std_dev_breakpoints(similarities, num_std=1.0):
    mean, std = np.mean(similarities), np.std(similarities)
    cutoff = mean - num_std * std
    return [i for i, s in enumerate(similarities) if s < cutoff]

In [8]:
## Structure-Aware Pre-Splitting (headings/clauses before semantic pass)

import re

def structure_split(text):
    """Split on markdown headers or numbered clauses before semantic chunking."""
    pattern = r"(?=\n#{1,3}\s)|(?=\n\d+\.\d+\s)"
    sections = re.split(pattern, text)
    return [s.strip() for s in sections if s.strip()]

def hybrid_chunk(text, embed_fn, similarity_threshold=0.75, max_chunk_tokens=300):
    """Structure-aware split first, then semantic chunking within any oversized section."""
    sections = structure_split(text)
    all_chunks = []
    for section in sections:
        sentences = re.split(r"(?<=[.!?])\s+", section)
        sentences = [s for s in sentences if s]
        if not sentences:
            continue
        if len(section.split()) <= max_chunk_tokens:
            all_chunks.append(section)
        else:
            all_chunks.extend(semantic_chunk(sentences, embed_fn, similarity_threshold, max_chunk_tokens))
    return all_chunks

In [9]:
## Evaluate Chunking Quality Against Human-Labeled Boundaries

def boundary_precision_recall(predicted_breaks, true_breaks, tolerance=1):
    """A predicted break counts as correct if within `tolerance` sentences of a true break."""
    matched_true = set()
    tp = 0
    for p in predicted_breaks:
        for t in true_breaks:
            if abs(p - t) <= tolerance and t not in matched_true:
                matched_true.add(t)
                tp += 1
                break
    precision = tp / len(predicted_breaks) if predicted_breaks else 0.0
    recall = tp / len(true_breaks) if true_breaks else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

def sweep_thresholds(similarities, true_breaks, thresholds):
    """Grid-search fixed thresholds to find the one maximizing F1 (ties to Q3's tuning answer)."""
    results = {}
    for t in thresholds:
        preds = fixed_threshold_breakpoints(similarities, t)
        results[t] = boundary_precision_recall(preds, true_breaks)
    return results

In [11]:
## Demo: Compare fixed-size vs semantic chunking on a toy audit-style paragraph

def fake_embed(sentence):
    """Deterministic toy embedding (stand-in for a real model) for offline demo purposes."""
    np.random.seed(abs(hash(sentence)) % (2**32))
    return np.random.rand(16)

sample_sentences = [
    "The contractor shall complete all deliverables by the agreed deadline.",
    "Failure to meet the deadline results in a penalty clause of 2% per week.",
    "The penalty is capped at 20% of the total contract value.",
    "Data privacy requirements mandate encryption of all client records.",
    "All personally identifiable information must be redacted before storage.",
    "Access to client records is restricted to authorized personnel only.",
]

similarities = compute_similarities(sample_sentences, fake_embed)
print("Sentence-to-sentence similarities:", [round(s, 2) for s in similarities])

chunks = semantic_chunk(sample_sentences, fake_embed, similarity_threshold=0.75, max_chunk_tokens=300)
for i, c in enumerate(chunks):
    print(f"Chunk {i+1}: {c}\n")

## Sweep thresholds against a labeled true breakpoint (ties sweep_thresholds into the demo)

# True topic shift is between sentence idx 2 ("...20% of contract value.") and idx 3
# ("Data privacy requirements...") -> that's boundary index 2 in the similarities list
true_breaks = [2] # the real topic shift is at boundary index 2
thresholds = [0.5, 0.6, 0.7, 0.75, 0.8, 0.9]

sweep_results = sweep_thresholds(similarities, true_breaks, thresholds)
for t, metrics in sweep_results.items():
    print(f"threshold={t}: precision={metrics['precision']:.2f}, "
          f"recall={metrics['recall']:.2f}, f1={metrics['f1']:.2f}")

best_threshold = max(sweep_results, key=lambda t: sweep_results[t]["f1"])
print(f"\nBest threshold by F1: {best_threshold}")

Sentence-to-sentence similarities: [0.79, 0.72, 0.68, 0.75, 0.72]
Chunk 1: The contractor shall complete all deliverables by the agreed deadline. Failure to meet the deadline results in a penalty clause of 2% per week.

Chunk 2: The penalty is capped at 20% of the total contract value.

Chunk 3: Data privacy requirements mandate encryption of all client records.

Chunk 4: All personally identifiable information must be redacted before storage.

Chunk 5: Access to client records is restricted to authorized personnel only.

threshold=0.5: precision=0.00, recall=0.00, f1=0.00
threshold=0.6: precision=0.00, recall=0.00, f1=0.00
threshold=0.7: precision=1.00, recall=1.00, f1=1.00
threshold=0.75: precision=0.25, recall=1.00, f1=0.40
threshold=0.8: precision=0.20, recall=1.00, f1=0.33
threshold=0.9: precision=0.20, recall=1.00, f1=0.33

Best threshold by F1: 0.7


In [15]:
import faiss

# Build embeddings from existing notebook variables/functions
doc_embeddings = np.array([fake_embed(text) for text in chunks], dtype="float32")

# Match FAISS index dimension to embedding size
dimension = doc_embeddings.shape[1]

# HNSW index config
index = faiss.IndexHNSWFlat(dimension, 32)
index.hnsw.efConstruction = 200

index.add(doc_embeddings)  # doc_embeddings: (n_docs, dimension) float32 array

# efSearch: query-time search depth (higher = better recall, slower query)
index.hnsw.efSearch = 64
k = min(5, len(chunks))
query_text = "data privacy"  # reuse existing notebook variable as the query
query_embedding = np.array(fake_embed(query_text), dtype="float32")
distances, indices = index.search(query_embedding.reshape(1, -1), k=k)

print("indices:", indices)
print("distances:", distances)

indices: [[3 0 1 4 2]]
distances: [[2.7494144 3.054368  3.413408  4.361437  4.482122 ]]


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_community.llms import HuggingFacePipeline  # wraps your local model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(chunks, embeddings)

llm = HuggingFacePipeline.from_model_id(model_id="your-local-model-path", task="text-generation")

# MultiQueryRetriever: LLM generates paraphrased queries, retrieves for each, dedupes results
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=llm,
)
results = multi_query_retriever.invoke("can the vendor be let go early?")

In [ ]:
from langchain.docstore.document import Document
from langchain.chains import RetrievalQAWithSourcesChain

# attach source metadata at ingestion time
docs = [
    Document(page_content=chunk, metadata={"source": "Contract_A.pdf", "clause": "4.2"})
    for chunk in chunks
]
vectorstore = FAISS.from_documents(docs, embeddings)

qa_chain = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm, retriever=vectorstore.as_retriever()
)
response = qa_chain.invoke({"question": "What is the termination penalty?"})
print(response["answer"], response["sources"])  # sources pulled straight from metadata

In [ ]:
from langchain.indexes import SQLRecordManager, index

record_manager = SQLRecordManager("faiss/audit_chunks", db_url="sqlite:///record_manager.db")
record_manager.create_schema()

# Only new/changed documents (by content hash) get re-embedded and added; unchanged ones are skipped
index(docs, record_manager, vectorstore, cleanup="incremental", source_id_key="source")

In [2]:
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

# Cache embeddings so repeated/identical chunks or queries aren't re-embedded
store = LocalFileStore("./embedding_cache")
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, store, namespace="all-MiniLM-L6-v2")

# Cache LLM responses so identical queries skip a full generation call
set_llm_cache(SQLiteCache(database_path="llm_cache.db"))

ImportError: cannot import name 'CacheBackedEmbeddings' from 'langchain.embeddings' (/Users/monusingh/work-share/code-blogs-articles/.venv/lib/python3.12/site-packages/langchain/embeddings/__init__.py)

In [ ]:
from sentence_transformers import CrossEncoder

nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-base")

def check_groundedness(context, answer):
    scores = nli_model.predict([(context, answer)])  # returns [contradiction, entailment, neutral] logits
    label = scores.argmax()
    return label == 1  # True if "entailment" (answer is supported by context)

In [1]:
JUDGE_PROMPT = """You are evaluating an AI assistant's answer to an audit-related question.

Question: {query}
Retrieved Context: {context}
Generated Answer: {answer}

Score each criterion from 1-5 and respond in JSON:
- faithfulness: Is every claim in the answer supported by the context? (1=unsupported, 5=fully supported)
- completeness: Does the answer address all parts of the question? (1=incomplete, 5=fully complete)
- relevance: Is the answer focused on what was asked, without irrelevant content? (1=off-topic, 5=fully relevant)

Respond with only: {{"faithfulness": X, "completeness": X, "relevance": X, "reasoning": "..."}}
"""